In [ ]:
import pandas as pd, glob, os

src = r"C:\Users\HI-55\olist_raw"  
out = ["# Data Profiling Report\n"]

for f in sorted(glob.glob(f"{src}/*.csv")):
    name = os.path.basename(f)
    df = pd.read_csv(f)
    out.append(f"\n## {name}\n")
    out.append(f"- Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
    out.append(f"- Duplicate rows: {df.duplicated().sum():,}\n")
    out.append("| Column | Type | Missing | Missing % | Unique |")
    out.append("|---|---|---|---|---|")
    for c in df.columns:
        m = df[c].isna().sum()
        out.append(f"| {c} | {df[c].dtype} | {m:,} | {m/len(df)*100:.1f}% | {df[c].nunique():,} |")

os.makedirs("docs/data", exist_ok=True)
open("../docs/data/data_profiling_report.md", "w", encoding="utf-8").write("\n".join(out))
print("Done")

Done


In [4]:
orders    = pd.read_csv(f"{src}/olist_orders_dataset.csv")
items     = pd.read_csv(f"{src}/olist_order_items_dataset.csv")
customers = pd.read_csv(f"{src}/olist_customers_dataset.csv")
products  = pd.read_csv(f"{src}/olist_products_dataset.csv")
sellers   = pd.read_csv(f"{src}/olist_sellers_dataset.csv")

print("items.order_id not in orders:",   (~items.order_id.isin(orders.order_id)).sum())
print("items.product_id not in products:", (~items.product_id.isin(products.product_id)).sum())
print("items.seller_id not in sellers:",   (~items.seller_id.isin(sellers.seller_id)).sum())
print("orders.customer_id not in customers:", (~orders.customer_id.isin(customers.customer_id)).sum())
print("duplicate order_id in orders:", orders.order_id.duplicated().sum())
print("duplicate (order_id, order_item_id) in items:", items.duplicated(["order_id","order_item_id"]).sum())

items.order_id not in orders: 0
items.product_id not in products: 0
items.seller_id not in sellers: 0
orders.customer_id not in customers: 0
duplicate order_id in orders: 0
duplicate (order_id, order_item_id) in items: 0


In [5]:
payments = pd.read_csv(f"{src}/olist_order_payments_dataset.csv")
trans    = pd.read_csv(f"{src}/olist_product_category_name_translation.csv") if False else pd.read_csv(f"{src}/product_category_name_translation.csv")

print(orders.order_status.value_counts())
print(payments.payment_type.value_counts())
print("orders without items:", (~orders.order_id.isin(items.order_id)).sum())
print("orders without payments:", (~orders.order_id.isin(payments.order_id)).sum())
missing_trans = set(products.product_category_name.dropna()) - set(trans.product_category_name)
print("categories without translation:", missing_trans)
print(orders[orders.order_id.isin(items.order_id) == False].order_status.value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64
orders without items: 775
orders without payments: 1
categories without translation: {'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


In [10]:
import glob, os

out_dir = "../docs/data"
os.makedirs(out_dir, exist_ok=True)

out = ["# Source Schema\n", "Types as read from the raw CSV files (Bronze, unchanged).\n"]
for f in sorted(glob.glob(f"{src}/*.csv")):
    df = pd.read_csv(f)
    out.append(f"\n## {os.path.basename(f)}\n")
    out.append("| Column | Source Type | Nullable |")
    out.append("|---|---|---|")
    for c in df.columns:
        out.append(f"| {c} | {df[c].dtype} | {'Yes' if df[c].isna().any() else 'No'} |")

path = os.path.join(out_dir, "source_schema.md")
open(path, "w", encoding="utf-8").write("\n".join(out))
print("saved to:", os.path.abspath(path))

saved to: c:\Users\HI-55\cloud-retail-data-platform\docs\data\source_schema.md
